In [88]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import skew
from scipy.stats import entropy

In [89]:
def calculate_entropy(signal, bins=30):
    hist, _ = np.histogram(signal, bins=bins, density=True)
    hist += 1e-12  # уникнути лог(0)
    return entropy(hist)


In [90]:
def interquartile_range(signal):
    return np.percentile(signal, 75) - np.percentile(signal, 25)

In [91]:
def mean_absolute_deviation(signal):
    return np.mean(np.abs(signal - np.mean(signal)))


In [92]:
def skewness(signal):
    return skew(signal)


In [93]:
def root_mean_square(signal):
    return np.sqrt(np.mean(np.square(signal)))

In [71]:
activities = ['idle','running','stairs','walking']
active_data = []

for active in activities:
    path = Path('data')/active
    csv_path = list(path.glob('*.csv'))
    df = pd.concat((pd.read_csv(file) for file in csv_path))
    df['active'] = active
    active_data.append(df)

data = pd.concat(active_data,ignore_index=True)

In [ ]:
window_size = 208
step_size = 104 

features = [] 

index = 0
while index + window_size <= data.shape[0]:
    window = data[index:index + window_size]
    
    x = window['accelerometer_X']
    y = window['accelerometer_Y']
    z = window['accelerometer_Z']

    features.append({
        'active': window['active'].mode()[0],
        'x_max': x.max(),
        'x_min': x.min(),
        'x_entropy': calculate_entropy(x),
        'x_iqr': interquartile_range(x),
        'y_max':y.max(),
        'y_argmin': np.argmin(y),
        'y_mad': mean_absolute_deviation(y),
        'y_median': y.median(),
        'y_skew':skew(y),
        'y_std': y.std(),
        'y_rmse':root_mean_square(y),
        'z_skew': skew(z)
    })
    
    index += step_size 


features_df = pd.DataFrame(features)
features_df


,active,x_max,x_min,x_entropy,x_iqr,y_max,y_argmin,y_mad,y_median,y_skew,y_std,y_rmse,z_skew
0,idle,5.099650,-0.909797,1.340327,0.512359,4.616021,2,0.147914,-0.033519,8.996577,0.450944,0.450342,-7.029484
1,idle,0.560243,0.268151,2.562424,0.043096,0.086191,14,0.033614,-0.023942,-1.044892,0.050729,0.058665,0.625035
2,idle,0.560243,-0.124498,2.113969,0.052673,0.244209,22,0.064556,-0.023942,1.166106,0.094950,0.094722,0.400344
3,idle,0.560243,-0.129287,2.021739,0.038307,0.244209,2,0.071363,-0.023942,1.016259,0.100738,0.100730,0.267572
4,idle,0.560243,-0.129287,1.643749,0.028731,0.239420,12,0.031589,-0.023942,1.135746,0.057144,0.060654,0.369367
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1858,walking,15.188815,-6.947973,2.554642,3.964798,-1.872266,9,3.389165,-10.208876,-0.261344,4.269769,10.868086,-0.526986
1859,walking,4.711789,-14.159308,2.659839,3.969587,1.580173,17,3.397767,-10.302250,-0.128650,4.223811,10.934541,-1.708733
1860,walking,7.192182,-14.159308,2.707825,4.965574,5.214572,148,3.635988,-10.208876,-0.267085,4.676320,11.227567,-1.175814
1861,walking,7.192182,-9.155427,3.019861,5.243302,5.214572,44,3.913270,-10.395624,-0.407202,5.065325,11.778179,0.611668


In [98]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1863 entries, 0 to 1862
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   active     1863 non-null   object 
 1   x_max      1863 non-null   float64
 2   x_min      1863 non-null   float64
 3   x_entropy  1863 non-null   float64
 4   x_iqr      1863 non-null   float64
 5   y_max      1863 non-null   float64
 6   y_argmin   1863 non-null   int64  
 7   y_mad      1863 non-null   float64
 8   y_median   1863 non-null   float64
 9   y_skew     1863 non-null   float64
 10  y_std      1863 non-null   float64
 11  y_rmse     1863 non-null   float64
 12  z_skew     1863 non-null   float64
dtypes: float64(11), int64(1), object(1)
memory usage: 189.3+ KB


In [108]:
X = features_df.drop(columns = 'active')
y = features_df['active']

In [109]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.3,random_state=42)

In [110]:
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

In [111]:
model_svc = SVC()
model_svc.fit(X_train_scaler,y_train)

SVC()

In [113]:
model_svc.__dict__

{'decision_function_shape': 'ovr',
 'break_ties': False,
 'kernel': 'rbf',
 'degree': 3,
 'gamma': 'scale',
 'coef0': 0.0,
 'tol': 0.001,
 'C': 1.0,
 'nu': 0.0,
 'epsilon': 0.0,
 'shrinking': True,
 'probability': False,
 'cache_size': 200,
 'class_weight': None,
 'verbose': False,
 'max_iter': -1,
 'random_state': None,
 '_sparse': False,
 'n_features_in_': 12,
 'class_weight_': array([1., 1., 1., 1.]),
 'classes_': array(['idle', 'running', 'stairs', 'walking'], dtype=object),
 '_gamma': 0.08333333333333333,
 'support_': array([  63,   68,   72,   97,  118,  139,  147,  176,  198,  201,  212,
         290,  394,  450,  462,  504,  515,  532,  535,  542,  621,  784,
         793,  830,  870,  888,  897,  898,  905,  939,  944,  970,  996,
        1005, 1028, 1039, 1151, 1182, 1207, 1237, 1256, 1284,   66,   73,
         117,  164,  197,  211,  213,  241,  252,  309,  352,  359,  418,
         480,  561,  724,  860,  884, 1072, 1105, 1167, 1172, 1258,    2,
          38,   58,   65,   

In [115]:
y_predict_svc = model_svc.predict(X_test_scaler)

In [117]:
score = accuracy_score(y_test,y_predict_svc)
score

0.9874776386404294

In [119]:
cls_rep = classification_report(y_test,y_predict_svc)
print(cls_rep)

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00        79
     running       1.00      1.00      1.00       299
      stairs       1.00      0.56      0.72        16
     walking       0.96      1.00      0.98       165

    accuracy                           0.99       559
   macro avg       0.99      0.89      0.92       559
weighted avg       0.99      0.99      0.99       559



In [121]:
model_tree = RandomForestClassifier(random_state=42)
model_tree.fit(X_train_scaler,y_train)

RandomForestClassifier(random_state=42)

In [122]:
model_tree.__dict__

{'estimator': DecisionTreeClassifier(),
 'n_estimators': 100,
 'estimator_params': ('criterion',
  'max_depth',
  'min_samples_split',
  'min_samples_leaf',
  'min_weight_fraction_leaf',
  'max_features',
  'max_leaf_nodes',
  'min_impurity_decrease',
  'random_state',
  'ccp_alpha',
  'monotonic_cst'),
 'bootstrap': True,
 'oob_score': False,
 'n_jobs': None,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False,
 'class_weight': None,
 'max_samples': None,
 'criterion': 'gini',
 'max_depth': None,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'min_weight_fraction_leaf': 0.0,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'monotonic_cst': None,
 'ccp_alpha': 0.0,
 'n_features_in_': 12,
 '_n_samples': 1304,
 'n_outputs_': 1,
 'classes_': array(['idle', 'running', 'stairs', 'walking'], dtype=object),
 'n_classes_': 4,
 '_n_samples_bootstrap': 1304,
 'estimator_': DecisionTreeClassifier(),
 'estimators_': [DecisionTreeClassifier(max_features='sqr

In [123]:
y_predict_forest = model_tree.predict(X_test_scaler)

In [124]:
score_forest = accuracy_score(y_test,y_predict_forest)
score_forest

0.998211091234347

In [125]:
cls_rep_forest = classification_report(y_test,y_predict_forest)
print(cls_rep_forest)

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00        79
     running       1.00      1.00      1.00       299
      stairs       1.00      0.94      0.97        16
     walking       0.99      1.00      1.00       165

    accuracy                           1.00       559
   macro avg       1.00      0.98      0.99       559
weighted avg       1.00      1.00      1.00       559



По accuracy_score ми бачимо що обидві моделі добре справляються з класифікацією і різниця між цими показниками мінімальна. Але якщо поглянути на інші метрики то бачимо що модель на методі SVM справляється набагато гірше. Це видно по метриці f1-score для stairs. Модель на методі SVM менш точно прогнозує що це саме підйом по сходах у звязку малою кількістю наданих даних.